# Model 3: SentenceTransformer End-to-End Fine-tuning

**Architecture:** all-MiniLM-L6-v2 (fine-tuned, 22M params) → mean pooling (384-dim) → regression head  
**Key difference from Model 1:** Encoder is NOT frozen — trained end-to-end on price data  

```
Model 1: all-MiniLM-L6-v2 (FROZEN) → pre-computed 384-dim → DNN head
Model 3: all-MiniLM-L6-v2 (FINE-TUNED) → batch-wise 384-dim → DNN head
```

**Hypothesis:** Frozen SentTrans was optimized for semantic similarity, not price prediction.  
Fine-tuning teaches the encoder to attend to brand names, materials, and category keywords.

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

In [ ]:
from pricer.items import Item
from pricer.senttrans_e2e_model import SentTransE2ERunner
from pricer.evaluator import evaluate, plot_training_history

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_full")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

Loads all-MiniLM-L6-v2 with requires_grad=True. No pre-computation — encoder runs in each batch.

In [ ]:
runner = SentTransE2ERunner(train, val[:1000])
runner.setup(batch_size=128)

## 3. Train

Max 15 epochs, early stopping patience=3.  
Discriminative LR: encoder=5e-5 (preserve pretrained), head=1e-3.

In [ ]:
history = runner.train(epochs=15, patience=3, warmup_steps=500)

## 4. Training History

In [ ]:
plot_training_history(history, title="SentTrans E2E Fine-tuning")

## 5. Evaluate on 200 Test Samples

In [ ]:
evaluate(runner.inference, test)

## 6. Save Model Weights

In [ ]:
runner.save("senttrans_e2e_model.pth")
print("Saved to senttrans_e2e_model.pth")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  ${sample.price:.2f}")
print(f"Predict: ${pred:.2f}")
print(f"Error:   ${abs(pred - sample.price):.2f}")